# SilkRoute Benchmark — 00: Schema Validation (Hardcoded Target)
This notebook validates the **generated SilkRoute benchmark dataset** against the **target tables + ERD** described in the SilkRoute case-study document.

Unlike the earlier version, the **target schema is hardcoded** here (based on the document), so the notebook:
- does **not** depend on doc parsing
- can be used in CI to enforce a stable contract

## 0. Setup

In [2]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUT = "../silkroute_benchmark_out"  # adjust if needed

def load_table(name):
    return pd.read_parquet(f"{OUT}/{name}.parquet")

def pretty_bool(b): 
    return "✅" if bool(b) else "❌"


## 1. Hardcoded target schema (tables + columns)
Target entities and fields are taken from the SilkRoute core entity table in the reference doc.
Key points include:
- `store_id` and `salesperson_id` are **nullable** for online transactions
- promotions are **product-level**, while inventory/sales/returns are **SKU-level**


In [3]:
# ---- TARGET TABLE SCHEMA (HARD-CODED) ----
# Source: SilkRoute case study "Core Business Entities (ContinuumAI MVP)" table.
TARGET_SCHEMAS = {
    # Dimension-ish
    "channels": {
        "required": ["channel_type", "channel_name"],
        "primary_key": ["channel_type"],
    },
    "stores": {
        "required": ["store_id", "store_name", "city", "region", "store_type"],
        "primary_key": ["store_id"],
    },
    "customers": {
        "required": ["customer_id", "segment", "city", "region", "first_purchase_date"],
        "primary_key": ["customer_id"],
    },
    "salespeople": {
        "required": ["salesperson_id", "name", "role", "store_id"],
        "primary_key": ["salesperson_id"],
        "foreign_keys": [("store_id", "stores", "store_id")],
    },
    "products": {
        "required": ["product_id", "product_name", "brand", "category", "subcategory", "status"],
        "primary_key": ["product_id"],
    },
    "product_variants_skus": {
        "required": ["sku_id", "product_id", "size", "color", "base_price", "active_flag"],
        "primary_key": ["sku_id"],
        "foreign_keys": [("product_id", "products", "product_id")],
    },
    "variant_attributes": {
        "required": ["sku_id", "attribute_name", "attribute_value", "attribute_type"],
        "primary_key": None,  # flexible attributes: (sku_id, attribute_name) is typical but not required by doc
        "foreign_keys": [("sku_id", "product_variants_skus", "sku_id")],
    },
    "category_attribute_definitions": {
        "required": ["category", "attribute_name", "attribute_type", "required_flag"],
        "primary_key": None,  # (category, attribute_name) typical
    },
    "promotions": {
        "required": ["promo_id", "product_id", "promo_type", "start_date", "end_date", "discount_type"],
        "primary_key": ["promo_id"],
        "foreign_keys": [("product_id", "products", "product_id")],
    },
    # Facts
    "transactions": {
        "required": ["transaction_id", "transaction_ts", "channel_type", "store_id", "customer_id", "salesperson_id", "payment_method", "total_amount"],
        "primary_key": ["transaction_id"],
        "foreign_keys": [
            ("channel_type", "channels", "channel_type"),
            ("customer_id", "customers", "customer_id"),
            # store_id and salesperson_id are conditional (store only)
        ],
        "conditional_nullability": {
            "online": {"store_id": True, "salesperson_id": True},
            "store": {"store_id": False, "salesperson_id": False},
        }
    },
    "transaction_lines": {
        "required": ["line_id", "transaction_id", "sku_id", "quantity", "unit_price", "discount", "line_total"],
        "primary_key": ["line_id"],
        "foreign_keys": [
            ("transaction_id", "transactions", "transaction_id"),
            ("sku_id", "product_variants_skus", "sku_id"),
        ],
    },
    "inventory_snapshots": {
        "required": ["snapshot_date", "store_id", "sku_id", "stock_on_hand", "stock_on_order"],
        "primary_key": None,  # typically (snapshot_date, store_id, sku_id)
        "foreign_keys": [
            ("sku_id", "product_variants_skus", "sku_id"),
            # store_id can include DC pseudo-store 'S000' (implementation detail)
        ],
        "store_id_allows_dc": True,
    },
    "returns": {
        "required": ["return_id", "transaction_id", "sku_id", "return_reason", "refund_amount"],
        "primary_key": ["return_id"],
        "foreign_keys": [
            ("transaction_id", "transactions", "transaction_id"),
            ("sku_id", "product_variants_skus", "sku_id"),
        ],
    },
}

TARGET_TABLES = list(TARGET_SCHEMAS.keys())
TARGET_TABLES


['channels',
 'stores',
 'customers',
 'salespeople',
 'products',
 'product_variants_skus',
 'variant_attributes',
 'category_attribute_definitions',
 'promotions',
 'transactions',
 'transaction_lines',
 'inventory_snapshots',
 'returns']

## 2. Load generated tables

In [4]:
tables = {}
missing_files = []
for t in TARGET_TABLES:
    try:
        tables[t] = load_table(t)
    except Exception as e:
        missing_files.append((t, str(e)))

missing_files, {k: v.shape for k,v in tables.items()}


([],
 {'channels': (2, 2),
  'stores': (6, 5),
  'customers': (2500, 5),
  'salespeople': (26, 4),
  'products': (90, 6),
  'product_variants_skus': (150, 6),
  'variant_attributes': (246, 4),
  'category_attribute_definitions': (11, 4),
  'promotions': (12, 6),
  'transactions': (11402, 8),
  'transaction_lines': (29924, 7),
  'inventory_snapshots': (55650, 5),
  'returns': (1287, 5)})

## 3. Column contract validation (missing / extra)

In [5]:
rows = []
for t, spec in TARGET_SCHEMAS.items():
    if t not in tables:
        rows.append({"table": t, "status": "MISSING_TABLE", "missing_cols": "", "extra_cols": "", "ok": False})
        continue
    exp = spec["required"]
    act = list(tables[t].columns)
    missing = [c for c in exp if c not in act]
    extra = [c for c in act if c not in exp]
    rows.append({
        "table": t,
        "status": "OK" if not missing else "MISSING_COLUMNS",
        "missing_cols": ", ".join(missing),
        "extra_cols": ", ".join(extra),
        "ok": len(missing)==0,
    })

col_report = pd.DataFrame(rows).sort_values("table")
col_report


,table,status,missing_cols,extra_cols,ok
7,category_attribute_definitions,OK,,,True
0,channels,OK,,,True
2,customers,OK,,,True
11,inventory_snapshots,OK,,,True
5,product_variants_skus,OK,,,True
4,products,OK,,,True
8,promotions,OK,,,True
12,returns,OK,,,True
3,salespeople,OK,,,True
1,stores,OK,,,True


## 4. Primary key validation (uniqueness)

In [6]:
pk_rows = []
for t, spec in TARGET_SCHEMAS.items():
    if t not in tables:
        continue
    pk = spec.get("primary_key")
    if not pk:
        pk_rows.append({"table": t, "pk": None, "unique": None, "notes": "no PK required by doc"})
        continue
    df = tables[t]
    unique = df.duplicated(pk).sum() == 0
    pk_rows.append({"table": t, "pk": ",".join(pk), "unique": unique, "notes": ""})

pk_report = pd.DataFrame(pk_rows).sort_values("table")
pk_report.assign(result=lambda d: d["unique"].map(lambda x: "—" if x is None else pretty_bool(x)))


,table,pk,unique,notes,result
7,category_attribute_definitions,NaN,None,no PK required by doc,—
0,channels,channel_type,True,,✅
2,customers,customer_id,True,,✅
11,inventory_snapshots,NaN,None,no PK required by doc,—
5,product_variants_skus,sku_id,True,,✅
4,products,product_id,True,,✅
8,promotions,promo_id,True,,✅
12,returns,return_id,True,,✅
3,salespeople,salesperson_id,True,,✅
1,stores,store_id,True,,✅


## 5. Foreign key validation (ERD / relationship integrity)

In [7]:
fk_rows = []
for t, spec in TARGET_SCHEMAS.items():
    if t not in tables:
        continue
    for fk in spec.get("foreign_keys", []):
        col, parent_table, parent_col = fk
        if parent_table not in tables:
            fk_rows.append({"table": t, "fk_col": col, "parent": parent_table, "status": "MISSING_PARENT_TABLE", "ok": False})
            continue
        if col not in tables[t].columns:
            fk_rows.append({"table": t, "fk_col": col, "parent": parent_table, "status": "MISSING_FK_COL", "ok": False})
            continue
        ok = tables[t][col].isin(tables[parent_table][parent_col]).all()
        fk_rows.append({"table": t, "fk_col": col, "parent": f"{parent_table}.{parent_col}", "status": "OK" if ok else "FK_VIOLATION", "ok": ok})

# Special-case: inventory_snapshots.store_id can include DC 'S000'
if "inventory_snapshots" in tables and "stores" in tables and "store_id" in tables["inventory_snapshots"].columns:
    valid = set(tables["stores"]["store_id"]).union({"S000"})
    ok = tables["inventory_snapshots"]["store_id"].isin(valid).all()
    fk_rows.append({"table": "inventory_snapshots", "fk_col": "store_id", "parent": "stores.store_id | S000", "status": "OK" if ok else "FK_VIOLATION", "ok": ok})

fk_report = pd.DataFrame(fk_rows).sort_values(["table","fk_col"])
fk_report.assign(result=lambda d: d["ok"].map(pretty_bool))


,table,fk_col,parent,status,ok,result
8,inventory_snapshots,sku_id,product_variants_skus.sku_id,OK,True,✅
11,inventory_snapshots,store_id,stores.store_id | S000,OK,True,✅
1,product_variants_skus,product_id,products.product_id,OK,True,✅
3,promotions,product_id,products.product_id,OK,True,✅
10,returns,sku_id,product_variants_skus.sku_id,OK,True,✅
9,returns,transaction_id,transactions.transaction_id,OK,True,✅
0,salespeople,store_id,stores.store_id,OK,True,✅
7,transaction_lines,sku_id,product_variants_skus.sku_id,OK,True,✅
6,transaction_lines,transaction_id,transactions.transaction_id,OK,True,✅
4,transactions,channel_type,channels.channel_type,OK,True,✅


## 6. Conditional nullability rules (online vs store)
The SilkRoute spec explicitly says `store_id` and `salesperson_id` are nullable for **online** transactions, but required for **store** transactions.


In [19]:
# tx = tables.get("transactions")
tx = pd.read_parquet("../silkroute_benchmark_out/transactions.parquet")
print(tx.dtypes)

if tx is None:
    print("transactions table missing")
else:
    tx = tx.copy()
    tx["channel_type"] = tx["channel_type"].astype(str)
    is_online = tx["channel_type"].eq("online")
    is_store = tx["channel_type"].eq("store")

    report = {
        "online store_id null_rate (expected high)": tx.loc[is_online, "store_id"].isna().mean(),
        "online salesperson_id null_rate (expected high)": tx.loc[is_online, "salesperson_id"].isna().mean(),
        "store store_id non-null_rate (expected 1.0)": 1 - tx.loc[is_store, "store_id"].isna().mean(),
        "store salesperson_id non-null_rate (expected 1.0)": 1 - tx.loc[is_store, "salesperson_id"].isna().mean(),
        "channel_type values": sorted(tx["channel_type"].unique().tolist()),
    }
    pd.Series(report)


transaction_id        str
transaction_ts        str
channel_type          str
store_id              str
customer_id           str
salesperson_id        str
payment_method        str
total_amount      float64
dtype: object


## 7. Type sanity (high-signal columns)
The document defines fields, not strict types. Here we validate the expected **semantic types** for key analytics columns.

In [18]:
import pandas as pd
from pandas.api.types import is_datetime64_any_dtype, is_numeric_dtype

def is_datetime_series(s):
    """Robust datetime dtype check (works with pandas extension dtypes)."""
    try:
        return is_datetime64_any_dtype(s)
    except Exception:
        return False

def is_numeric_series(s):
    """Robust numeric dtype check (works with pandas extension dtypes)."""
    try:
        return is_numeric_dtype(s)
    except Exception:
        return False

checks = []

def check(t, col, pred, expectation):
    if t not in tables:
        checks.append([t, col, "MISSING_TABLE", False, expectation])
        return
    df = tables[t]
    if col not in df.columns:
        checks.append([t, col, "MISSING_COL", False, expectation])
        return
    ok = bool(pred(df[col]))
    checks.append([t, col, str(df[col].dtype), ok, expectation])

check("transactions", "transaction_ts", is_datetime_series, "transaction_ts should be datetime")
check("promotions", "start_date", is_datetime_series, "start_date should be datetime")
check("promotions", "end_date", is_datetime_series, "end_date should be datetime")
check("inventory_snapshots", "snapshot_date", is_datetime_series, "snapshot_date should be datetime")

check("transactions", "total_amount", is_numeric_series, "total_amount should be numeric")

for col in ["quantity","unit_price","discount","line_total"]:
    check("transaction_lines", col, is_numeric_series, f"{col} should be numeric")

for col in ["stock_on_hand","stock_on_order"]:
    check("inventory_snapshots", col, is_numeric_series, f"{col} should be numeric")

if "returns" in tables:
    check("returns", "refund_amount", is_numeric_series, "refund_amount should be numeric")

type_report = pd.DataFrame(checks, columns=["table","column","dtype","ok","expectation"])
type_report.assign(result=lambda d: d["ok"].map(pretty_bool))


,table,column,dtype,ok,expectation,result
0,transactions,transaction_ts,str,False,transaction_ts should be datetime,❌
1,promotions,start_date,str,False,start_date should be datetime,❌
2,promotions,end_date,str,False,end_date should be datetime,❌
3,inventory_snapshots,snapshot_date,str,False,snapshot_date should be datetime,❌
4,transactions,total_amount,float64,True,total_amount should be numeric,✅
5,transaction_lines,quantity,int64,True,quantity should be numeric,✅
6,transaction_lines,unit_price,float64,True,unit_price should be numeric,✅
7,transaction_lines,discount,float64,True,discount should be numeric,✅
8,transaction_lines,line_total,float64,True,line_total should be numeric,✅
9,inventory_snapshots,stock_on_hand,int64,True,stock_on_hand should be numeric,✅


## 8. Structural profiling (shape + missingness + cardinality)

In [10]:
def profile_table(df: pd.DataFrame, name: str):
    prof = pd.DataFrame({
        "column": df.columns,
        "dtype": [str(df[c].dtype) for c in df.columns],
        "null_rate": [df[c].isna().mean() for c in df.columns],
        "n_unique": [df[c].nunique(dropna=True) for c in df.columns],
        "sample_values": [
            ", ".join([str(x) for x in df[c].dropna().unique()[:3]])
            if df[c].nunique(dropna=True) <= 20 else ""
            for c in df.columns
        ]
    })
    prof.insert(0, "table", name)
    return prof

profiles = pd.concat([profile_table(df, name) for name, df in tables.items()], ignore_index=True)
profiles.head(20)


,table,column,dtype,null_rate,n_unique,sample_values
0,channels,channel_type,str,0.0,2,"store, online"
1,channels,channel_name,str,0.0,2,"SilkRoute (Brick-and-Mortar), SilkRoute Online..."
2,stores,store_id,str,0.0,6,"S001, S002, S003"
3,stores,store_name,str,0.0,3,"SilkRoute Karachi 1, SilkRoute Lahore 2, SilkR..."
4,stores,city,str,0.0,3,"Karachi, Lahore, Islamabad"
5,stores,region,str,0.0,3,"Sindh, Punjab, ICT"
6,stores,store_type,str,0.0,3,"flagship, mall, high_street"
7,customers,customer_id,str,0.0,2500,
8,customers,segment,str,0.0,5,"one_time, price_sensitive, high_return"
9,customers,city,str,0.0,3,"Karachi, Islamabad, Lahore"


In [11]:
# Columns with high missingness (useful for spotting structural issues)
profiles[profiles["null_rate"] >= 0.30].sort_values(["table","null_rate"], ascending=[True, False])


,table,column,dtype,null_rate,n_unique,sample_values
24,product_variants_skus,size,str,0.513333,5,"L, XS, M"
45,transactions,store_id,str,0.324504,6,"S006, S002, S005"
47,transactions,salesperson_id,str,0.324504,26,


## 9. Alignment scorecard

In [12]:
# Scorecard from the three hard constraints: required columns, PKs (where defined), FKs
col_ok = col_report.set_index("table")["ok"].to_dict()
pk_ok = {r["table"]: (True if r["unique"] is None else bool(r["unique"])) for _, r in pk_report.iterrows()}
fk_ok = fk_report.groupby("table")["ok"].all().to_dict() if len(fk_report) else {}

score_rows = []
for t in TARGET_TABLES:
    score_rows.append({
        "table": t,
        "required_columns_ok": col_ok.get(t, False),
        "pk_ok": pk_ok.get(t, True),
        "fk_ok": fk_ok.get(t, True),
    })

score = pd.DataFrame(score_rows)
score["overall_ok"] = score["required_columns_ok"] & score["pk_ok"] & score["fk_ok"]
score.assign(
    required_columns_ok=score["required_columns_ok"].map(pretty_bool),
    pk_ok=score["pk_ok"].map(pretty_bool),
    fk_ok=score["fk_ok"].map(pretty_bool),
    overall_ok=score["overall_ok"].map(pretty_bool),
)


,table,required_columns_ok,pk_ok,fk_ok,overall_ok
0,channels,✅,✅,✅,✅
1,stores,✅,✅,✅,✅
2,customers,✅,✅,✅,✅
3,salespeople,✅,✅,✅,✅
4,products,✅,✅,✅,✅
5,product_variants_skus,✅,✅,✅,✅
6,variant_attributes,✅,✅,✅,✅
7,category_attribute_definitions,✅,✅,✅,✅
8,promotions,✅,✅,✅,✅
9,transactions,✅,✅,✅,✅


## 10. Optional: Operating scale checks

In [13]:
scale = {}
if "stores" in tables: scale["stores"] = len(tables["stores"])
if "product_variants_skus" in tables: scale["skus"] = len(tables["product_variants_skus"])
if "customers" in tables: scale["customers"] = len(tables["customers"])
if "transactions" in tables: scale["transactions"] = len(tables["transactions"])
if "transaction_lines" in tables: scale["transaction_lines"] = len(tables["transaction_lines"])
if "salespeople" in tables: scale["salespeople"] = len(tables["salespeople"])
if "returns" in tables: scale["returns"] = len(tables["returns"])

pd.Series(scale)


stores                   6
skus                   150
customers             2500
transactions         11402
transaction_lines    29924
salespeople             26
returns               1287
dtype: int64

In [14]:
if "transactions" in tables and "returns" in tables and len(tables["transactions"])>0:
    ret_rate = tables["returns"]["transaction_id"].nunique() / tables["transactions"]["transaction_id"].nunique()
    print("Return rate (tx-level):", round(ret_rate, 4), "(", round(ret_rate*100,2), "% )")


Return rate (tx-level): 0.0995 ( 9.95 % )
